In [1]:

# EfficientNet-B0 — Power-of-Two


import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0
from sklearn.model_selection import train_test_split

device = torch.device("cpu")

# Dataset & transforms 
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

# Stratified split 85/10/5 
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for c in np.unique(targets):
    idxs = np.where(targets == c)[0]
    tr, tmp = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_dataset, val_dataset, test_dataset = (
    Subset(dataset, train_idx),
    Subset(dataset, val_idx),
    Subset(dataset, test_idx)
)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")

# Load FP32 EfficientNet-B0 
ckpt = torch.load("efficientnet_b0_weights.pth", map_location="cpu")
ckpt = {k.replace("_orig_mod.", ""): v for k, v in ckpt.items()}

num_classes = len(dataset.classes)
model_fp32 = efficientnet_b0(weights=None)
model_fp32.classifier[1] = nn.Linear(model_fp32.classifier[1].in_features, num_classes)
missing, unexpected = model_fp32.load_state_dict(ckpt, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
model_fp32.eval().to(device)
print("FP32 EfficientNet-B0 loaded")

#  Power-of-two quantizer 
def power_of_two_quantize(weight, n_bits=8):
    w = weight.detach().clone()
    wmin, wmax = w.min(), w.max()
    scale = (wmax - wmin) / (2 ** n_bits - 1 + 1e-12)
    pow2_scale = 2 ** torch.round(torch.log2(scale + 1e-12))
    zero_point = torch.round(-wmin / (pow2_scale + 1e-12))
    w_q = torch.round(w / (pow2_scale + 1e-12) + zero_point)
    w_q = torch.clamp(w_q, 0, 2 ** n_bits - 1)
    q_w = pow2_scale * (w_q - zero_point)
    return q_w

def apply_power_of_two(model, n_bits=8):
    m = copy.deepcopy(model)
    for name, p in m.named_parameters():
        if "weight" in name and p is not None:
            with torch.no_grad():
                p.copy_(power_of_two_quantize(p.data, n_bits=n_bits))
    return m

model_pot = apply_power_of_two(model_fp32, n_bits=8)
model_pot.eval()
print("Applied Power-of-Two Quantization")

# Eval / size / runtime
def evaluate(model, loader):
    model.eval()
    corr, tot = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            out = model(x.to(device))
            preds = out.argmax(1).cpu()
            corr += (preds == y).sum().item()
            tot  += y.size(0)
    return 100.0 * corr / tot

def get_model_size(m, path="__tmp__.pth"):
    torch.save(m.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return mb

def benchmark(m, loader, num_batches=50):
    m.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            _ = m(x.to(device))
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0) / num_batches * 1000.0, (t1 - t0) / imgs * 1000.0

acc_fp32 = evaluate(model_fp32, test_loader)
acc_pot  = evaluate(model_pot,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"Power-of-Two Accuracy: {acc_pot:.2f}%")

sz_fp32 = get_model_size(model_fp32)
sz_pot  = get_model_size(model_pot)
print(f"FP32 size: {sz_fp32:.2f} MB | Power-of-Two size: {sz_pot:.2f} MB")

b_fp32, i_fp32 = benchmark(model_fp32, test_loader)
b_pot,  i_pot  = benchmark(model_pot,  test_loader)
print(f"FP32 Inference: {b_fp32:.2f} ms/batch | {i_fp32:.2f} ms/image")
print(f"Power-of-Two Inference: {b_pot:.2f} ms/batch | {i_pot:.2f} ms/image")



Total images: 100000, Classes: 200
DataLoaders ready
Missing keys: []
Unexpected keys: []
FP32 EfficientNet-B0 loaded
Applied Power-of-Two Quantization
FP32 Accuracy: 74.60%
Power-of-Two Accuracy: 64.26%
FP32 size: 17.35 MB | Power-of-Two size: 17.35 MB
FP32 Inference: 1834.41 ms/batch | 14.33 ms/image
Power-of-Two Inference: 1758.22 ms/batch | 13.74 ms/image
